In [ ]:
# cell 1
# Mount Google Drive, define paths, and configure PyRAG inference.

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import time
import shlex
import shutil
import socket
import subprocess
import gc
import signal
import traceback
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/final_project"

PYRAG_WORK_DIR = f"{PROJECT_DIR}/pyrag"
REPO_DIR = "/content/PyRAG"
INDEX_ROOT = f"{PYRAG_WORK_DIR}/indexes"
PREPARED_Q_DIR = f"{PYRAG_WORK_DIR}/prepared_questions"
CUSTOM_SCRIPTS_DIR = f"{PYRAG_WORK_DIR}/custom_scripts"

PHASE1_MANIFEST = f"{PYRAG_WORK_DIR}/phase1_manifest.json"

RUN_OUTPUT_DIR = f"{PYRAG_WORK_DIR}/runs"
RAW_OUTPUT_DIR = f"{PYRAG_WORK_DIR}/raw_pyrag_outputs"
FINAL_OUTPUT_DIR = PYRAG_WORK_DIR

for p in [PYRAG_WORK_DIR, RUN_OUTPUT_DIR, RAW_OUTPUT_DIR, FINAL_OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

DATASETS = {
    "hotpotqa": {
        "questions": f"{PROJECT_DIR}/hotpotqa_dev_2017wiki_1000_converted.json",
        "prepared_questions": f"{PREPARED_Q_DIR}/hotpotqa_shuffled.json",
        "index_dir": f"{INDEX_ROOT}/hotpotqa",
        "raw_jsonl": f"{RAW_OUTPUT_DIR}/hotpotqa_pyrag_raw_results.jsonl",
        "progress_jsonl": f"{RAW_OUTPUT_DIR}/hotpotqa_evidence_progress.jsonl",
        "final_evidence": f"{FINAL_OUTPUT_DIR}/hotpotqa_evidence.json",
    },
    "2wikimultihopqa": {
        "questions": f"{PROJECT_DIR}/2wikimultihopqa_dev_2020wiki_1000_converted.json",
        "prepared_questions": f"{PREPARED_Q_DIR}/2wikimultihopqa_shuffled.json",
        "index_dir": f"{INDEX_ROOT}/2wikimultihopqa",
        "raw_jsonl": f"{RAW_OUTPUT_DIR}/2wikimultihopqa_pyrag_raw_results.jsonl",
        "progress_jsonl": f"{RAW_OUTPUT_DIR}/2wikimultihopqa_evidence_progress.jsonl",
        "final_evidence": f"{FINAL_OUTPUT_DIR}/2wikimultihopqa_evidence.json",
    },
}

# Fair-comparison retrieval budget.
# PyRAG first retrieves 6 RAW chunks per original question.
# Then, after duplicate removal, the notebook tries to fill the final evidence
# up to 6 UNIQUE chunks using only the original question.
#
# Important:
# - answer/supports/gold evidence are never used for retrieval.
# - final evidence_chunk should ideally contain 6 unique chunks.
# - if the retriever cannot find enough non-duplicate candidates, the notebook
#   keeps fewer than 6 and reports it later, without stopping the full run.
DEFAULT_TOPK = 6

TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION = 6
TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION = 6

MAX_TOTAL_RETRIEVED_CHUNKS_PER_QUESTION = TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION

# Post-dedup unique fill.
# If final evidence after duplicate removal has fewer than 6 chunks,
# cell 11 retrieves extra candidates with sample["question"] only,
# skips duplicates, and appends only unique chunks.
ENABLE_POST_DEDUP_UNIQUE_FILL = True
POST_DEDUP_FILL_CANDIDATE_TOPK = 50

# Disable topk=10 adaptive expansion from the repository runner.
# A topk=10 retry would make this method unfair against fixed top-6 baselines.
DISABLE_ADAPTIVE_TOPK_EXPANSION = True
ADAPTIVE_RETRY_TOPK = TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION

# Keep exactly the models used by the PyRAG repository recommendation.
PLAN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
INSTRUCT_MODEL = "Qwen/Qwen2.5-7B-Instruct"

PLAN_PORT = 8336
INSTRUCT_PORT = 8337
RETRIEVER_PORT = 8008

PLAN_BASE_URL = f"http://127.0.0.1:{PLAN_PORT}/v1"
INSTRUCT_BASE_URL = f"http://127.0.0.1:{INSTRUCT_PORT}/v1"

# Two 7B BF16 vLLM servers on A100 80GB.
# 0.40 each is usually safe. If OOM happens, reduce to 0.36.
GPU_MEMORY_UTILIZATION_PLAN = 0.40
GPU_MEMORY_UTILIZATION_INSTRUCT = 0.40

# Context budget.
# Retrieval evidence is controlled to 6 raw chunks first,
# then final evidence is filled toward 6 unique chunks after dedup.
MAX_MODEL_LEN = 8192

# Keep one sequence at a time to control memory.
MAX_NUM_SEQS = 1

# Match batching budget with the model context length.
MAX_NUM_BATCHED_TOKENS = 8192

# Keep retriever on CPU so vLLM owns GPU memory.
RETRIEVER_DEVICE = "cpu"

RUN_HOTPOTQA = True
RUN_2WIKIMULTIHOPQA = True

# Use None for full 1000 samples.
MAX_SAMPLES_PER_DATASET = None

# Save final JSON every N completed samples.
SAVE_EVERY = 10

PLAN_SERVER_LOG_PATH = Path(f"{RUN_OUTPUT_DIR}/vllm_plan_{PLAN_PORT}.log")
INSTRUCT_SERVER_LOG_PATH = Path(f"{RUN_OUTPUT_DIR}/vllm_instruct_{INSTRUCT_PORT}.log")
PLAN_SERVER_PID_PATH = Path(f"{RUN_OUTPUT_DIR}/vllm_plan_{PLAN_PORT}.pid")
INSTRUCT_SERVER_PID_PATH = Path(f"{RUN_OUTPUT_DIR}/vllm_instruct_{INSTRUCT_PORT}.pid")

# Keep TensorFlow out of transformers/vLLM imports.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print("Project dir:", PROJECT_DIR)
print("PyRAG work dir:", PYRAG_WORK_DIR)
print("Plan model:", PLAN_MODEL)
print("Instruct model:", INSTRUCT_MODEL)
print("Plan base URL:", PLAN_BASE_URL)
print("Instruct base URL:", INSTRUCT_BASE_URL)
print("MAX_MODEL_LEN:", MAX_MODEL_LEN)
print("MAX_NUM_BATCHED_TOKENS:", MAX_NUM_BATCHED_TOKENS)
print("DEFAULT_TOPK:", DEFAULT_TOPK)
print("TARGET RAW retrieved chunks per question:", TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION)
print("TARGET UNIQUE evidence chunks per question:", TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)
print("Post-dedup unique fill enabled:", ENABLE_POST_DEDUP_UNIQUE_FILL)
print("Post-dedup fill candidate topk:", POST_DEDUP_FILL_CANDIDATE_TOPK)
print("Adaptive topk expansion disabled:", DISABLE_ADAPTIVE_TOPK_EXPANSION)
print("Retrieval uses question only; answer/supports are not used for retrieval.")

subprocess.run(["nvidia-smi"], check=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/final_project
PyRAG work dir: /content/drive/MyDrive/final_project/pyrag
Plan model: Qwen/Qwen2.5-Coder-7B-Instruct
Instruct model: Qwen/Qwen2.5-7B-Instruct
Plan base URL: http://127.0.0.1:8336/v1
Instruct base URL: http://127.0.0.1:8337/v1
MAX_MODEL_LEN: 8192
MAX_NUM_BATCHED_TOKENS: 8192
DEFAULT_TOPK: 6
TARGET RAW retrieved chunks per question: 6
TARGET UNIQUE evidence chunks per question: 6
Post-dedup unique fill enabled: True
Post-dedup fill candidate topk: 50
Adaptive topk expansion disabled: True
Retrieval uses question only; answer/supports are not used for retrieval.


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
# cell 2
# Minimal Colab setup for PyRAG + vLLM.
#
# Important:
# - Do not uninstall torch.
# - Do not pin old vLLM.
# - Do not pin old transformers.
# - Remove only torchvision because it is broken in this Colab runtime and is not needed for Qwen text-only serving.

import os
import sys
import subprocess
import importlib.util

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

def run(cmd, check=True):
    # Run command and show it.
    print("\n$ " + " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check)

def run_capture(cmd, check=False):
    # Run command and show stdout/stderr.
    print("\n$ " + " ".join(str(x) for x in cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=os.environ.copy(),
    )
    if result.stdout.strip():
        print("\n--- STDOUT ---")
        print(result.stdout)
    if result.stderr.strip():
        print("\n--- STDERR ---")
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError("Command failed: " + " ".join(str(x) for x in cmd))
    return result

print("Python:", sys.version)
run(["nvidia-smi"], check=False)

# The current error is:
# RuntimeError: operator torchvision::nms does not exist
# This means torchvision is incompatible with the current torch.
# Qwen2.5 text-only vLLM serving does not need torchvision.
run_capture([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)

# Install uv only if missing.
if importlib.util.find_spec("uv") is None:
    run([sys.executable, "-m", "pip", "-q", "install", "-U", "uv"], check=True)
else:
    print("uv already available.")

# Install only missing/small runtime packages.
# Do not force reinstall torch/vLLM/transformers.
run([
    "uv", "pip", "install", "--system",
    "openai",
    "requests",
    "tqdm",
    "jsonschema",
    "psutil",
    "accelerate",
    "safetensors",
    "fastapi",
    "uvicorn[standard]",
    "faiss-cpu",
    "sentence-transformers",
], check=True)

# Install vLLM only if it is missing.
vllm_check = run_capture([
    sys.executable,
    "-c",
    (
        "import os; "
        "os.environ['USE_TF']='0'; "
        "os.environ['USE_TORCH']='1'; "
        "os.environ['TRANSFORMERS_NO_TF']='1'; "
        "import vllm; "
        "print('vllm import OK:', vllm.__version__)"
    )
], check=False)

if vllm_check.returncode != 0:
    print("vLLM is missing or broken. Installing vLLM with uv for the current Colab CUDA stack.")
    run([
        "uv", "pip", "install", "--system", "-U",
        "vllm",
        "--torch-backend=auto",
    ], check=True)
else:
    print("vLLM is already importable. Skipping vLLM install.")

# Final checks.
run_capture([
    sys.executable,
    "-c",
    (
        "import os; "
        "os.environ['USE_TF']='0'; "
        "os.environ['USE_TORCH']='1'; "
        "os.environ['TRANSFORMERS_NO_TF']='1'; "
        "import torch, transformers, vllm; "
        "print('torch:', torch.__version__); "
        "print('torch cuda:', torch.version.cuda); "
        "print('cuda available:', torch.cuda.is_available()); "
        "print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); "
        "print('transformers:', transformers.__version__); "
        "print('vllm:', vllm.__version__)"
    )
], check=True)

run(["which", "vllm"], check=True)
run(["vllm", "--version"], check=False)
run(["nvidia-smi"], check=False)

print("Cell 2 completed.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

$ nvidia-smi

$ /usr/bin/python3 -m pip uninstall -y torchvision

--- STDOUT ---
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130

uv already available.

$ uv pip install --system openai requests tqdm jsonschema psutil accelerate safetensors fastapi uvicorn[standard] faiss-cpu sentence-transformers

$ /usr/bin/python3 -c import os; os.environ['USE_TF']='0'; os.environ['USE_TORCH']='1'; os.environ['TRANSFORMERS_NO_TF']='1'; import vllm; print('vllm import OK:', vllm.__version__)

--- STDOUT ---
vllm import OK: 0.23.0

vLLM is already importable. Skipping vLLM install.

$ /usr/bin/python3 -c import os; os.environ['USE_TF']='0'; os.environ['USE_TORCH']='1'; os.environ['TRANSFORMERS_NO_TF']='1'; import torch, transformers, vllm; print('torch:', torch.__version__); print('torch cuda:', torch.version.cuda); print('cuda available:', torch.

In [ ]:
# cell 4
# Clone or update the requested PyRAG repository.

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ali-mohmmadi/PyRAG.git"

if not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=False)
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])

sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + ":" + os.environ.get("PYTHONPATH", "")

print("Repository ready:", REPO_DIR)
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], check=False)

Repository ready: /content/PyRAG


CompletedProcess(args=['git', '-C', '/content/PyRAG', 'log', '-1', '--oneline'], returncode=0)

In [ ]:
# cell 5
# Validate Phase 1 outputs.

import json
from pathlib import Path

required_phase1_files = [
    PHASE1_MANIFEST,
    f"{CUSTOM_SCRIPTS_DIR}/local_e5_retriever_server.py",
]

for dataset_name, cfg in DATASETS.items():
    required_phase1_files.extend([
        cfg["questions"],
        cfg["prepared_questions"],
        f"{cfg['index_dir']}/corpus.jsonl",
        f"{cfg['index_dir']}/index.faiss",
        f"{cfg['index_dir']}/index_meta.json",
    ])

missing = [p for p in required_phase1_files if not Path(p).exists()]
if missing:
    print("Missing Phase 1 files:")
    for p in missing:
        print("  -", p)
    raise FileNotFoundError("Phase 1 is incomplete.")

with open(PHASE1_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Loaded Phase 1 manifest:", PHASE1_MANIFEST)
print("Embedding model:", manifest.get("embedding_model"))

for dataset_name, cfg in DATASETS.items():
    with open(f"{cfg['index_dir']}/index_meta.json", "r", encoding="utf-8") as f:
        meta = json.load(f)

    with open(cfg["prepared_questions"], "r", encoding="utf-8") as f:
        prepared_questions = json.load(f)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Index vectors:", meta.get("num_vectors"))
    print("Prepared shuffled questions:", len(prepared_questions))
    print("First question:", prepared_questions[0].get("question"))

print("Phase 1 validation passed.")

Loaded Phase 1 manifest: /content/drive/MyDrive/final_project/pyrag/phase1_manifest.json
Embedding model: intfloat/e5-base-v2
Dataset: hotpotqa
Index vectors: 35029
Prepared shuffled questions: 1000
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Dataset: 2wikimultihopqa
Index vectors: 12685
Prepared shuffled questions: 1000
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
Phase 1 validation passed.


In [ ]:
# cell 6
# Process, port, and log utilities.

import os
import gc
import time
import socket
import subprocess
from pathlib import Path

import psutil
import requests
from openai import OpenAI

try:
    import torch
except Exception:
    torch = None

process_registry = {}

def is_port_open(port, host="127.0.0.1"):
    # Check whether a TCP port is open.
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

def tail_log(path, n=80):
    # Read last n log lines.
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="ignore").splitlines()[-n:])

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def kill_leftover_vllm_processes():
    # Kill old vLLM serve processes.
    current_pid = os.getpid()
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            pid = p.info["pid"]
            if pid == current_pid:
                continue
            cmdline = " ".join(p.info.get("cmdline") or [])
            if "vllm" in cmdline and "serve" in cmdline:
                kill_process_tree(pid)
                print("Killed leftover vLLM process:", pid)
        except Exception:
            pass

def cleanup_gpu_memory():
    # Free local GPU cache.
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(3)
    subprocess.run(["nvidia-smi"], check=False)

def wait_for_vllm_server(proc, port, base_url, model_name, log_path, max_wait_sec=1800):
    # Wait for a vLLM server and show live logs.
    ready = False
    sleep_sec = 5
    print_every_sec = 60
    start = time.perf_counter()
    last_print = -print_every_sec

    for _ in range(max_wait_sec // sleep_sec):
        elapsed = int(time.perf_counter() - start)

        return_code = proc.poll()
        if return_code is not None:
            print(f"vLLM process exited. Return code: {return_code}")
            print("\n=== Last vLLM log lines ===")
            print(tail_log(log_path, n=200))
            raise RuntimeError("vLLM server crashed or exited during startup.")

        try:
            h = requests.get(f"http://localhost:{port}/health", timeout=5)
            if h.status_code == 200:
                m = requests.get(f"{base_url}/models", timeout=10)
                if m.status_code == 200:
                    model_ids = [x["id"] for x in m.json()["data"]]
                    print("vLLM server is ready.")
                    print("Base URL:", base_url)
                    print("Models:", model_ids)

                    if model_name not in model_ids:
                        print("WARNING: expected model not found exactly.")
                        print("Expected:", model_name)

                    ready = True
                    break
        except Exception:
            pass

        if elapsed - last_print >= print_every_sec:
            last_print = elapsed
            print(f"Waiting... {elapsed}s")
            recent = tail_log(log_path, n=20)
            if recent.strip():
                print(recent)
            else:
                print("No log output yet.")
            print("-" * 100)
            subprocess.run(["nvidia-smi"], check=False)

        time.sleep(sleep_sec)

    if not ready:
        print("\n=== Last vLLM log lines ===")
        print(tail_log(log_path, n=200))
        raise RuntimeError("vLLM server did not become ready before timeout.")

    client = OpenAI(api_key="EMPTY", base_url=base_url, timeout=3600)

    test = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a test service."},
            {"role": "user", "content": "Reply exactly: OK"},
        ],
        max_tokens=8,
        temperature=0.0,
        extra_body={
            "top_k": 20,
            "repetition_penalty": 1.05,
        },
    )

    print("Test response:", test.choices[0].message.content)
    return client

print("Utility functions ready.")

Utility functions ready.


In [ ]:
# cell 7
# Start the two Qwen2.5 vLLM servers.
#
# This is text-only Qwen2.5 serving:
# - no Qwen3 reasoning parser
# - no old vLLM pinning
# - no torchvision dependency
# - live log like your working Qwen3.5 notebook

def start_vllm_server(
    name,
    model_name,
    port,
    base_url,
    log_path,
    pid_path,
    gpu_memory_utilization,
):
    # Stop old PID from this notebook.
    pid_path = Path(pid_path)
    log_path = Path(log_path)

    if pid_path.exists():
        old_pid = pid_path.read_text().strip()
        if old_pid:
            kill_process_tree(old_pid)

    if is_port_open(port):
        print(f"Port {port} is open. Killing leftover vLLM processes.")
        kill_leftover_vllm_processes()
        time.sleep(5)

    if is_port_open(port):
        raise RuntimeError(f"Port still occupied: {port}")

    cmd = [
        "vllm", "serve", model_name,

        "--host", "0.0.0.0",
        "--port", str(port),

        "--served-model-name", model_name,

        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(gpu_memory_utilization),

        "--language-model-only",

        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

        "--enable-prefix-caching",
        "--generation-config", "vllm",
        "--dtype", "bfloat16",
        "--trust-remote-code",
    ]

    server_env = os.environ.copy()
    server_env["CUDA_VISIBLE_DEVICES"] = "0"
    server_env["TOKENIZERS_PARALLELISM"] = "false"

    # Do not import TensorFlow through transformers.
    server_env["USE_TF"] = "0"
    server_env["USE_TORCH"] = "1"
    server_env["TRANSFORMERS_NO_TF"] = "1"
    server_env["TF_CPP_MIN_LOG_LEVEL"] = "3"

    # Same defensive setting as your successful Qwen3.5 notebook.
    server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

    # Current Colab reports CUDA 13.0.
    server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"

    # A100 is Ampere SM 8.0.
    server_env["TORCH_CUDA_ARCH_LIST"] = "8.0"

    print("=" * 100)
    print(f"Starting {name} vLLM server")
    print("Model:", model_name)
    print("Port:", port)
    print("Base URL:", base_url)
    print("Log:", log_path)
    print("\nCommand:")
    print(" ".join(shlex.quote(x) for x in cmd))

    print("\nImportant environment variables:")
    for k in [
        "CUDA_VISIBLE_DEVICES",
        "VLLM_USE_FLASHINFER_SAMPLER",
        "VLLM_MAIN_CUDA_VERSION",
        "TORCH_CUDA_ARCH_LIST",
        "USE_TF",
        "USE_TORCH",
        "TRANSFORMERS_NO_TF",
    ]:
        print(f"{k}={server_env.get(k)}")

    log_path.write_text("", encoding="utf-8")
    log_file = open(log_path, "w", encoding="utf-8")

    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=server_env,
    )

    pid_path.write_text(str(proc.pid))
    process_registry[f"vllm_{name}"] = proc

    print("\nStarted vLLM server.")
    print("PID:", proc.pid)
    print("Log:", log_path)

    return proc

kill_leftover_vllm_processes()
cleanup_gpu_memory()

print("Starting Plan Agent server.")
plan_proc = start_vllm_server(
    name="plan",
    model_name=PLAN_MODEL,
    port=PLAN_PORT,
    base_url=PLAN_BASE_URL,
    log_path=PLAN_SERVER_LOG_PATH,
    pid_path=PLAN_SERVER_PID_PATH,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION_PLAN,
)

plan_client = wait_for_vllm_server(
    proc=plan_proc,
    port=PLAN_PORT,
    base_url=PLAN_BASE_URL,
    model_name=PLAN_MODEL,
    log_path=PLAN_SERVER_LOG_PATH,
    max_wait_sec=1800,
)

print("\nStarting Instruct server.")
instruct_proc = start_vllm_server(
    name="instruct",
    model_name=INSTRUCT_MODEL,
    port=INSTRUCT_PORT,
    base_url=INSTRUCT_BASE_URL,
    log_path=INSTRUCT_SERVER_LOG_PATH,
    pid_path=INSTRUCT_SERVER_PID_PATH,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION_INSTRUCT,
)

instruct_client = wait_for_vllm_server(
    proc=instruct_proc,
    port=INSTRUCT_PORT,
    base_url=INSTRUCT_BASE_URL,
    model_name=INSTRUCT_MODEL,
    log_path=INSTRUCT_SERVER_LOG_PATH,
    max_wait_sec=1800,
)

print("\nBoth vLLM servers are ready.")
subprocess.run(["nvidia-smi"], check=False)

Starting Plan Agent server.
Starting plan vLLM server
Model: Qwen/Qwen2.5-Coder-7B-Instruct
Port: 8336
Base URL: http://127.0.0.1:8336/v1
Log: /content/drive/MyDrive/final_project/pyrag/runs/vllm_plan_8336.log

Command:
vllm serve Qwen/Qwen2.5-Coder-7B-Instruct --host 0.0.0.0 --port 8336 --served-model-name Qwen/Qwen2.5-Coder-7B-Instruct --max-model-len 8192 --gpu-memory-utilization 0.4 --language-model-only --max-num-seqs 1 --max-num-batched-tokens 8192 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
CUDA_VISIBLE_DEVICES=0
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=8.0
USE_TF=0
USE_TORCH=1
TRANSFORMERS_NO_TF=1

Started vLLM server.
PID: 7355
Log: /content/drive/MyDrive/final_project/pyrag/runs/vllm_plan_8336.log
Waiting... 0s
No log output yet.
----------------------------------------------------------------------------------------------------
Waiting... 60s
(APIServer pid=7355)

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
# cell 8
# Final OpenAI-compatible API sanity check.

from openai import OpenAI

def sanity_check_openai_chat(base_url, model_name, label):
    client = OpenAI(api_key="EMPTY", base_url=base_url, timeout=3600)

    print("=" * 100)
    print("Checking:", label)
    print("Base URL:", base_url)
    print("Model:", model_name)

    models = client.models.list()
    model_ids = [m.id for m in models.data]
    print("Available models:", model_ids)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a concise assistant."},
            {"role": "user", "content": "Return exactly: OK"},
        ],
        max_tokens=8,
        temperature=0.0,
        extra_body={
            "top_k": 20,
            "repetition_penalty": 1.05,
        },
    )

    print("Response:", response.choices[0].message.content)

sanity_check_openai_chat(PLAN_BASE_URL, PLAN_MODEL, "Plan Agent")
sanity_check_openai_chat(INSTRUCT_BASE_URL, INSTRUCT_MODEL, "Decompose + Answer Agent")

print("Both vLLM endpoints passed sanity check.")

Checking: Plan Agent
Base URL: http://127.0.0.1:8336/v1
Model: Qwen/Qwen2.5-Coder-7B-Instruct
Available models: ['Qwen/Qwen2.5-Coder-7B-Instruct']
Response: OK
Checking: Decompose + Answer Agent
Base URL: http://127.0.0.1:8337/v1
Model: Qwen/Qwen2.5-7B-Instruct
Available models: ['Qwen/Qwen2.5-7B-Instruct']
Response: OK
Both vLLM endpoints passed sanity check.


In [ ]:
# cell 9
# Start and validate the local retriever server for each dataset.

RETRIEVER_SERVER_SCRIPT = f"{CUSTOM_SCRIPTS_DIR}/local_e5_retriever_server.py"
EMBED_MODEL_NAME = manifest.get("embedding_model", "intfloat/e5-base-v2")

def stop_retriever():
    proc = process_registry.get("retriever")
    if proc is not None and proc.poll() is None:
        kill_process_tree(proc.pid)
        process_registry["retriever"] = None
        print("Stopped previous retriever process.")

def wait_for_http_json(url, timeout_seconds=600, sleep_seconds=5):
    start = time.time()
    last_error = None

    while time.time() - start < timeout_seconds:
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                return r.json()
            last_error = f"HTTP {r.status_code}: {r.text[:300]}"
        except Exception as e:
            last_error = repr(e)

        time.sleep(sleep_seconds)

    raise TimeoutError(f"Endpoint not ready: {url}\nLast error: {last_error}")

def start_retriever_for_dataset(dataset_name):
    if dataset_name not in DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    stop_retriever()

    cfg = DATASETS[dataset_name]
    index_dir = cfg["index_dir"]

    log_path = f"{RUN_OUTPUT_DIR}/retriever_{dataset_name}.log"
    log_file = open(log_path, "a", encoding="utf-8")

    cmd = [
        sys.executable,
        RETRIEVER_SERVER_SCRIPT,
        "--index_dir", index_dir,
        "--model_name", EMBED_MODEL_NAME,
        "--device", RETRIEVER_DEVICE,
        "--host", "127.0.0.1",
        "--port", str(RETRIEVER_PORT),
    ]

    print("=" * 100)
    print("Starting retriever for:", dataset_name)
    print("Index dir:", index_dir)
    print("Command:", " ".join(shlex.quote(x) for x in cmd))
    print("Log:", log_path)

    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    process_registry["retriever"] = proc

    health = wait_for_http_json(
        f"http://127.0.0.1:{RETRIEVER_PORT}/health",
        timeout_seconds=600,
        sleep_seconds=5,
    )

    print("Retriever health:", health)
    return proc, health

def test_retriever(dataset_name, topk=5):
    with open(DATASETS[dataset_name]["prepared_questions"], "r", encoding="utf-8") as f:
        samples = json.load(f)

    question = samples[0]["question"]

    payload = {
        "queries": [question],
        "topk": topk,
        "return_scores": True,
    }

    r = requests.post(
        f"http://127.0.0.1:{RETRIEVER_PORT}/retrieve",
        json=payload,
        timeout=60,
    )
    r.raise_for_status()
    hits = r.json()["result"][0]

    print("Dataset:", dataset_name)
    print("Question:", question)
    print("Hits:", len(hits))
    for i, hit in enumerate(hits[:3], 1):
        doc = hit["document"]
        print("-" * 100)
        print(f"Hit {i} | score={hit.get('score'):.4f}")
        print("Title:", doc.get("title", ""))
        print("Text preview:", doc.get("text", "")[:500])

for ds in ["hotpotqa", "2wikimultihopqa"]:
    start_retriever_for_dataset(ds)
    test_retriever(ds, topk=DEFAULT_TOPK)

print("Both retrievers validated.")

Starting retriever for: hotpotqa
Index dir: /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa
Command: /usr/bin/python3 /content/drive/MyDrive/final_project/pyrag/custom_scripts/local_e5_retriever_server.py --index_dir /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa --model_name intfloat/e5-base-v2 --device cpu --host 127.0.0.1 --port 8008
Log: /content/drive/MyDrive/final_project/pyrag/runs/retriever_hotpotqa.log
Retriever health: {'status': 'ok', 'num_documents': 35029, 'index_vectors': 35029, 'model_name': 'intfloat/e5-base-v2', 'device': 'cpu'}
Dataset: hotpotqa
Question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Hits: 6
----------------------------------------------------------------------------------------------------
Hit 1 | score=0.8881
Title: Bedknobs and Broomsticks
Text preview: Bedknobs and Broomsticks is a 1971 British-American musical fantasy film produced by Walt Disney Productions and released by B

In [ ]:
# cell 10
# Import PyRAG and build a runner with a fixed six-raw-chunk retrieval budget.
#
# What stays unchanged:
# - PyRAG still decomposes the question.
# - PyRAG still asks the planning LLM to generate Python code.
# - The generated code still calls retrieve() and answer() in the normal PyRAG way.
#
# What is constrained for fair comparison:
# - Across one original question, retrieve steps should return exactly 6 RAW chunks.
# - If there is one retrieve call, it gets 6 chunks.
# - If there are two retrieve calls, they get 3 and 3 chunks.
# - If there are three retrieve calls, they get 2, 2, and 2 chunks.
# - If the executed retrieve calls return fewer than 6 raw chunks, a final fill retrieve
#   with the original question is appended to the execution log.
# - No error is raised if the final unique evidence_chunk has fewer than 6 chunks
#   because duplicates are removed later in cell 11.

import ast
from typing import Any, Callable, Dict, List, Optional, Tuple

from pyrag import (
    HttpRetrievalAgent,
    OpenAILLM,
    RAGProgramRunner,
    env_enable_thinking,
)
from pyrag.tools import answer_system_prompt_for_docs
from pyrag.utils import extract_answer_tag, format_docs_for_prompt

try:
    from pyrag.runner import MAX_FIX_ROUNDS
except Exception:
    MAX_FIX_ROUNDS = 3

# Larger but controlled output budgets.
PLAN_MAX_TOKENS = 1024
INSTRUCT_MAX_TOKENS = 768
LLM_TEMPERATURE = 0.7


def count_retrieve_calls_in_code(code: str) -> int:
    # Count static retrieve(...) call sites in the generated PyRAG program.
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return 1

    count = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            fn = node.func
            if isinstance(fn, ast.Name) and fn.id == "retrieve":
                count += 1

    return max(count, 1)


def allocate_raw_retrieval_budget(total_budget: int, planned_calls: int) -> List[int]:
    # Distribute exactly total_budget raw chunks across planned retrieve calls.
    #
    # Examples for total_budget=6:
    # - 1 retrieve call  -> [6]
    # - 2 retrieve calls -> [3, 3]
    # - 3 retrieve calls -> [2, 2, 2]
    # - 4 retrieve calls -> [2, 2, 1, 1]
    # - 5 retrieve calls -> [2, 1, 1, 1, 1]
    # - 6 retrieve calls -> [1, 1, 1, 1, 1, 1]
    # - more than 6 calls -> first 6 calls get 1; later calls get 0.
    total_budget = max(0, int(total_budget))
    planned_calls = max(1, int(planned_calls))

    if total_budget == 0:
        return [0 for _ in range(planned_calls)]

    if planned_calls >= total_budget:
        return [1 if i < total_budget else 0 for i in range(planned_calls)]

    base = total_budget // planned_calls
    remainder = total_budget % planned_calls

    return [
        base + (1 if i < remainder else 0)
        for i in range(planned_calls)
    ]


def count_retrieved_docs_in_execution_log(execution_log: list) -> int:
    # Count RAW documents returned by retrieve steps.
    # Duplicates are counted here because the retrieval budget is raw top-k.
    return sum(
        len(entry.get("docs", []))
        for entry in execution_log
        if entry.get("type") == "retrieve"
    )


def count_unique_retrieved_docs_in_execution_log(execution_log: list) -> int:
    # Count unique documents returned by retrieve steps.
    # This is only for reporting, not for stopping the run.
    seen = set()

    for entry in execution_log:
        if entry.get("type") != "retrieve":
            continue

        for doc in entry.get("docs", []):
            seen.add(str(doc).strip())

    return len(seen)


def trim_retrieve_docs_to_raw_budget(execution_log: list, target_raw_chunks: int) -> list:
    # Safety clamp: if any backend unexpectedly returns more than requested,
    # trim retrieve docs so the raw retrieval log never exceeds the target.
    remaining = int(target_raw_chunks)
    new_log = []

    for entry in execution_log:
        if entry.get("type") != "retrieve":
            new_log.append(entry)
            continue

        new_entry = dict(entry)
        docs = list(new_entry.get("docs", []))

        if remaining <= 0:
            new_entry["docs"] = []
            new_entry["topk"] = 0
        else:
            new_entry["docs"] = docs[:remaining]
            new_entry["topk"] = len(new_entry["docs"])
            remaining -= len(new_entry["docs"])

        new_log.append(new_entry)

    return new_log


def fill_missing_raw_retrieval_budget(
    result: Dict[str, Any],
    retrieval_agent,
    original_query: str,
    target_raw_chunks: int,
) -> Dict[str, Any]:
    # If the generated code executed fewer retrieve calls than expected,
    # or some retrieve call returned fewer docs than requested, append one
    # final retrieve step using the original question.
    #
    # This fill step is only for the retrieved evidence log.
    # It does not use gold evidence and does not change the already generated answer.
    execution_log = result.get("execution_log", [])
    current_raw = count_retrieved_docs_in_execution_log(execution_log)
    missing = int(target_raw_chunks) - int(current_raw)

    if missing <= 0:
        result["execution_log"] = trim_retrieve_docs_to_raw_budget(
            execution_log,
            target_raw_chunks,
        )
        return result

    try:
        fill_docs = retrieval_agent.retrieve(original_query, topk=missing)
    except Exception as e:
        # Do not crash the full dataset run because of the fill step.
        # Leave the result as-is and record the warning in the result.
        result["_raw_retrieval_fill_warning"] = (
            f"Could not fill missing raw retrieval chunks. "
            f"Missing={missing}. Error={type(e).__name__}: {e}"
        )
        return result

    fill_docs = list(fill_docs)[:missing]

    if fill_docs:
        execution_log.append({
            "step": len(execution_log) + 1,
            "type": "retrieve",
            "query": original_query,
            "requested_topk": missing,
            "allocated_topk": missing,
            "topk": len(fill_docs),
            "global_retrieval_budget": target_raw_chunks,
            "used_global_retrieval_budget": current_raw + len(fill_docs),
            "planned_retrieve_calls": result.get("planned_retrieve_calls"),
            "is_budget_fill": True,
            "docs": fill_docs,
        })

    result["execution_log"] = trim_retrieve_docs_to_raw_budget(
        execution_log,
        target_raw_chunks,
    )

    final_raw = count_retrieved_docs_in_execution_log(result["execution_log"])
    if final_raw < target_raw_chunks:
        result["_raw_retrieval_fill_warning"] = (
            f"Raw retrieval count is still below target after fill. "
            f"Target={target_raw_chunks}, raw={final_raw}."
        )

    return result


def make_budgeted_tools(
    retrieval_agent,
    llm,
    default_topk: int = 6,
    target_raw_retrieved_chunks: int = 6,
    planned_retrieve_calls: int = 1,
) -> Tuple[Callable, Callable, List[Dict[str, Any]]]:
    # This is intentionally very close to pyrag.tools.make_tools().
    # The only change is that retrieve() is wrapped so that the generated PyRAG
    # program receives a fixed raw top-k budget across one original question.
    execution_log: List[Dict[str, Any]] = []

    planned_retrieve_calls = max(1, int(planned_retrieve_calls))
    target_raw_retrieved_chunks = max(0, int(target_raw_retrieved_chunks))

    allocations = allocate_raw_retrieval_budget(
        total_budget=target_raw_retrieved_chunks,
        planned_calls=planned_retrieve_calls,
    )

    state = {
        "retrieve_call_idx": 0,
        "used_budget": 0,
        "carry_deficit": 0,
    }

    def retrieve(query: str, topk: int = default_topk) -> List[str]:
        state["retrieve_call_idx"] += 1
        idx = state["retrieve_call_idx"]

        try:
            requested_topk = max(0, int(topk))
        except Exception:
            requested_topk = int(default_topk)

        base_allocation = allocations[idx - 1] if idx <= len(allocations) else 0
        remaining_budget = target_raw_retrieved_chunks - state["used_budget"]

        # Carry previous deficit forward, but never exceed the remaining global budget.
        allocated_topk = min(
            max(0, remaining_budget),
            max(0, base_allocation + state["carry_deficit"]),
        )

        if allocated_topk > 0:
            docs = retrieval_agent.retrieve(query, topk=allocated_topk)
            docs = list(docs)[:allocated_topk]
        else:
            docs = []

        state["used_budget"] += len(docs)
        state["carry_deficit"] = max(0, allocated_topk - len(docs))

        execution_log.append({
            "step": len(execution_log) + 1,
            "type": "retrieve",
            "query": query,
            "requested_topk": requested_topk,
            "allocated_topk": allocated_topk,
            "topk": len(docs),
            "global_retrieval_budget": target_raw_retrieved_chunks,
            "used_global_retrieval_budget": state["used_budget"],
            "planned_retrieve_calls": planned_retrieve_calls,
            "retrieval_budget_allocations": allocations,
            "docs": docs,
        })

        return docs

    def answer(query: str, docs: Optional[List[str]] = None) -> str:
        if docs is None:
            docs = []

        user_prompt = (
            f"=== QUESTION ===\n"
            f"{query}\n"
            f"=== END QUESTION ===\n\n"
            f"=== RETRIEVED DOCUMENTS ===\n"
            f"{format_docs_for_prompt(docs)}\n"
            f"=== END DOCUMENTS ==="
        )

        result = llm.generate(answer_system_prompt_for_docs(docs), user_prompt)
        returned = extract_answer_tag(result)

        execution_log.append({
            "step": len(execution_log) + 1,
            "type": "answer",
            "query": query,
            "docs": docs,
            "answer_raw": result,
            "answer_returned": returned,
        })

        return returned

    return retrieve, answer, execution_log


class CappedRAGProgramRunner(RAGProgramRunner):
    # A notebook-local runner that preserves PyRAG's decomposition, planning,
    # generated-code execution, and answer flow, while enforcing a fixed
    # six-raw-chunk retrieval budget per original question.

    def __init__(
        self,
        llm,
        retrieval_agent,
        plan_llm=None,
        target_raw_retrieved_chunks: int = 6,
        disable_adaptive_topk_expansion: bool = True,
    ):
        super().__init__(
            llm=llm,
            plan_llm=plan_llm,
            retrieval_agent=retrieval_agent,
        )
        self.target_raw_retrieved_chunks = int(target_raw_retrieved_chunks)
        self.max_total_retrieved_chunks = int(target_raw_retrieved_chunks)
        self.disable_adaptive_topk_expansion = bool(disable_adaptive_topk_expansion)

    def _execute_code_with_fixes(
        self,
        query: str,
        code: str,
        topk: int,
    ) -> Tuple[Dict[str, Any], str]:
        for fix_round in range(MAX_FIX_ROUNDS + 1):
            planned_retrieve_calls = count_retrieve_calls_in_code(code)

            retrieve_fn, answer_fn, execution_log = make_budgeted_tools(
                retrieval_agent=self.retrieval_agent,
                llm=self.llm,
                default_topk=topk,
                target_raw_retrieved_chunks=self.target_raw_retrieved_chunks,
                planned_retrieve_calls=planned_retrieve_calls,
            )

            try:
                result = self.executor.execute(code, retrieve_fn, answer_fn, execution_log)
                result["planned_retrieve_calls"] = planned_retrieve_calls

                # Fill missing raw retrieval docs if the executed program returned fewer than 6.
                result = fill_missing_raw_retrieval_budget(
                    result=result,
                    retrieval_agent=self.retrieval_agent,
                    original_query=query,
                    target_raw_chunks=self.target_raw_retrieved_chunks,
                )

                total_raw_retrieved = count_retrieved_docs_in_execution_log(result["execution_log"])
                total_unique_retrieved = count_unique_retrieved_docs_in_execution_log(result["execution_log"])

                # Do not raise for raw count mismatch. Record it and continue.
                result["retrieval_cap"] = self.target_raw_retrieved_chunks
                result["retrieval_target"] = self.target_raw_retrieved_chunks
                result["num_retrieved_chunks_total"] = total_raw_retrieved
                result["num_unique_retrieved_chunks_total"] = total_unique_retrieved
                result["planned_retrieve_calls"] = planned_retrieve_calls
                result["retrieval_budget_allocations"] = allocate_raw_retrieval_budget(
                    self.target_raw_retrieved_chunks,
                    planned_retrieve_calls,
                )

                return result, code

            except RuntimeError as e:
                # Keep PyRAG's own code-fix behavior for actual execution errors.
                # This is not used for duplicate/unique evidence count issues.
                error_msg = str(e)

                if fix_round == MAX_FIX_ROUNDS:
                    raise

                print(
                    f"\n[Fix round {fix_round + 1}/{MAX_FIX_ROUNDS}] "
                    f"Execution error, asking LLM to fix...\n  {error_msg.splitlines()[0]}"
                )

                code = self.plan_agent.fix_code(
                    original_query=query,
                    failed_code=code,
                    error_msg=error_msg,
                )

                print("=== Fixed Code ===")
                print(code)

        raise RuntimeError("_execute_code_with_fixes: exhausted fix rounds without return")

    def run(self, query: str, topk: int = 6) -> Dict[str, Any]:
        sub_queries = self.decompose_agent.decompose(query)

        print("\n=== Sub-queries ===")
        for i, q in enumerate(sub_queries, 1):
            print(f"  {i}. {q}")

        code = self.plan_agent.generate_code(query, sub_queries)

        print("\n=== Generated Code ===")
        print(code)

        result, code = self._execute_code_with_fixes(query, code, topk)

        # Do not use PyRAG's topk=10 adaptive expansion in this fair-comparison run.
        retried_with_topk10 = False

        self._print_trace(result)

        print("\n=== Retrieval Budget ===")
        print("Planned retrieve calls:", result.get("planned_retrieve_calls"))
        print("Budget allocations:", result.get("retrieval_budget_allocations"))
        print(
            f"RAW retrieved chunks: {result.get('num_retrieved_chunks_total')} / "
            f"{self.target_raw_retrieved_chunks}"
        )
        print(
            f"Unique retrieved chunks before final evidence extraction: "
            f"{result.get('num_unique_retrieved_chunks_total')}"
        )

        if result.get("_raw_retrieval_fill_warning"):
            print("WARNING:", result["_raw_retrieval_fill_warning"])

        print("\n=== Final Answer ===")
        print(result["final_answer"])

        return {
            "original_query": query,
            "sub_queries": sub_queries,
            "generated_code": code,
            "execution_log": result["execution_log"],
            "variables": result.get("variables", {}),
            "final_answer": result["final_answer"],
            "retried_with_topk10": retried_with_topk10,
            "retrieval_cap": self.target_raw_retrieved_chunks,
            "retrieval_target": self.target_raw_retrieved_chunks,
            "num_retrieved_chunks_total": result.get("num_retrieved_chunks_total"),
            "num_unique_retrieved_chunks_total": result.get("num_unique_retrieved_chunks_total"),
            "planned_retrieve_calls": result.get("planned_retrieve_calls"),
            "retrieval_budget_allocations": result.get("retrieval_budget_allocations"),
            "_raw_retrieval_fill_warning": result.get("_raw_retrieval_fill_warning"),
        }


def build_pyrag_runner():
    # Decompose + Answer model.
    instruct_llm = OpenAILLM(
        model=INSTRUCT_MODEL,
        base_url=INSTRUCT_BASE_URL,
        temperature=LLM_TEMPERATURE,
        max_tokens=INSTRUCT_MAX_TOKENS,
        enable_thinking=env_enable_thinking(),
    )

    # Plan model.
    plan_llm = OpenAILLM(
        model=PLAN_MODEL,
        base_url=PLAN_BASE_URL,
        temperature=LLM_TEMPERATURE,
        max_tokens=PLAN_MAX_TOKENS,
        enable_thinking=env_enable_thinking(),
    )

    retrieval_agent = HttpRetrievalAgent(
        host="127.0.0.1",
        port=RETRIEVER_PORT,
        timeout=60,
    )

    runner = CappedRAGProgramRunner(
        llm=instruct_llm,
        plan_llm=plan_llm,
        retrieval_agent=retrieval_agent,
        target_raw_retrieved_chunks=TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION,
        disable_adaptive_topk_expansion=DISABLE_ADAPTIVE_TOPK_EXPANSION,
    )

    return runner


runner = build_pyrag_runner()

print("Fixed raw-top-6 PyRAG runner is ready.")
print("MAX_MODEL_LEN:", MAX_MODEL_LEN)
print("PLAN_MAX_TOKENS:", PLAN_MAX_TOKENS)
print("INSTRUCT_MAX_TOKENS:", INSTRUCT_MAX_TOKENS)
print("DEFAULT_TOPK per retrieve request:", DEFAULT_TOPK)
print("TARGET RAW retrieved chunks per question:", TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION)
print("Final evidence_chunk may be < 6 after duplicate removal.")
print("Adaptive topk expansion disabled:", DISABLE_ADAPTIVE_TOPK_EXPANSION)

Fixed raw-top-6 PyRAG runner is ready.
MAX_MODEL_LEN: 8192
PLAN_MAX_TOKENS: 1024
INSTRUCT_MAX_TOKENS: 768
DEFAULT_TOPK per retrieve request: 6
TARGET RAW retrieved chunks per question: 6
Final evidence_chunk may be < 6 after duplicate removal.
Adaptive topk expansion disabled: True


In [ ]:
# cell 11
# Helper functions to parse retrieved documents from PyRAG execution_log
# and build the final evidence object.
#
# Important:
# - Retrieval uses only sample["question"].
# - Gold fields like answer/supports/titles/docs/docs_chunks/title_chunks are never used for retrieval.
# - type/question/answer/supports are copied to the output only for JSON structure/evaluation.
# - evidence_chunk is first extracted from retrieve steps in execution_log.
# - Duplicate evidence chunks are removed.
# - If final unique evidence_chunk has fewer than 6 chunks, this cell tries to fill
#   the missing chunks by retrieving extra candidates with the original question only.
# - Fill candidates are also deduplicated, so duplicate fill chunks are never added.
# - If fewer than 6 unique chunks can be found, no error is raised.

import re
from tqdm.auto import tqdm

from pyrag import HttpRetrievalAgent

DOC_RE = re.compile(
    r"^Doc\s+\d+\s+\(Title:\s*(.*?)\)\s*\n(.*)$",
    flags=re.DOTALL,
)


def load_json_array(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON array: {path}")
    return data


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


def atomic_write_json(path, obj):
    path = Path(path)
    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

    tmp_path.replace(path)


def parse_doc_string(doc_string):
    text = str(doc_string).strip()
    match = DOC_RE.match(text)

    if match:
        title = match.group(1).strip()
        body = match.group(2).strip()
        return {
            "title": title,
            "text": body,
        }

    # Fallback for unexpected formats.
    lines = text.splitlines()
    if len(lines) >= 2:
        return {
            "title": lines[0].strip(),
            "text": "\n".join(lines[1:]).strip(),
        }

    return {
        "title": "",
        "text": text,
    }


def evidence_key(doc):
    # Uniqueness is defined by title + text.
    return (
        str(doc.get("title", "")).strip(),
        str(doc.get("text", "")).strip(),
    )


def extract_evidence_chunks_from_execution_log(
    execution_log,
    max_chunks=TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION,
    deduplicate=True,
):
    # Extract evidence chunks from retrieve steps only.
    evidence = []
    seen = set()
    max_chunks = max(0, int(max_chunks))

    for entry in execution_log:
        if entry.get("type") != "retrieve":
            continue

        for doc_string in entry.get("docs", []):
            if len(evidence) >= max_chunks:
                return evidence

            doc = parse_doc_string(doc_string)
            key = evidence_key(doc)

            if deduplicate and key in seen:
                continue

            seen.add(key)
            evidence.append(doc)

    return evidence


def format_doc_for_execution_log(doc):
    # Keep fill docs in the same rough string format as PyRAG retrieved docs.
    title = str(doc.get("title", "")).strip()
    text = str(doc.get("text", "")).strip()

    if title:
        return f"Doc 0 (Title: {title})\n{text}"

    return text


def retrieve_unique_fill_from_question(
    question,
    existing_evidence,
    missing_count,
    candidate_topk=POST_DEDUP_FILL_CANDIDATE_TOPK,
):
    # Retrieve extra candidates using only the original question.
    # Never use answer/supports/gold evidence.
    # Add only candidates that are not duplicate with existing evidence.
    missing_count = max(0, int(missing_count))
    candidate_topk = max(missing_count, int(candidate_topk))

    if missing_count <= 0:
        return [], [], None

    existing_keys = {evidence_key(doc) for doc in existing_evidence}
    added_docs = []
    added_raw_docs = []
    added_keys = set()

    retrieval_agent = HttpRetrievalAgent(
        host="127.0.0.1",
        port=RETRIEVER_PORT,
        timeout=60,
    )

    try:
        raw_candidates = retrieval_agent.retrieve(question, topk=candidate_topk)
    except Exception as e:
        warning = (
            f"Post-dedup unique fill failed. "
            f"missing={missing_count}, candidate_topk={candidate_topk}, "
            f"error={type(e).__name__}: {e}"
        )
        return [], [], warning

    for raw_doc in raw_candidates:
        doc = parse_doc_string(raw_doc)
        key = evidence_key(doc)

        if key in existing_keys:
            continue

        if key in added_keys:
            continue

        added_docs.append(doc)
        added_raw_docs.append(raw_doc)
        added_keys.add(key)

        if len(added_docs) >= missing_count:
            break

    if len(added_docs) < missing_count:
        warning = (
            f"Post-dedup unique fill could not reach target. "
            f"needed={missing_count}, added={len(added_docs)}, "
            f"candidate_topk={candidate_topk}."
        )
    else:
        warning = None

    return added_docs, added_raw_docs, warning


def build_final_evidence_item(sample, pyrag_result):
    # Step 1: Extract unique evidence from PyRAG retrieve steps.
    evidence_chunks = extract_evidence_chunks_from_execution_log(
        pyrag_result.get("execution_log", []),
        max_chunks=TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION,
        deduplicate=True,
    )

    initial_unique_count = len(evidence_chunks)
    fill_warning = None
    fill_added = 0

    # Step 2: If dedup made the final evidence shorter than 6,
    # fill using only the original question.
    if (
        ENABLE_POST_DEDUP_UNIQUE_FILL
        and initial_unique_count < TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION
    ):
        missing = TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION - initial_unique_count

        fill_docs, fill_raw_docs, fill_warning = retrieve_unique_fill_from_question(
            question=sample["question"],
            existing_evidence=evidence_chunks,
            missing_count=missing,
            candidate_topk=POST_DEDUP_FILL_CANDIDATE_TOPK,
        )

        if fill_docs:
            evidence_chunks.extend(fill_docs)
            fill_added = len(fill_docs)

            # Add the post-dedup fill step to the PyRAG trace for auditability.
            # This does not use answer/supports and does not change the generated answer.
            execution_log = pyrag_result.setdefault("execution_log", [])
            execution_log.append({
                "step": len(execution_log) + 1,
                "type": "retrieve",
                "query": sample["question"],
                "requested_topk": POST_DEDUP_FILL_CANDIDATE_TOPK,
                "topk": len(fill_raw_docs),
                "is_post_dedup_unique_fill": True,
                "target_unique_evidence_chunks": TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION,
                "initial_unique_evidence_chunks": initial_unique_count,
                "final_unique_evidence_chunks": len(evidence_chunks),
                "docs": fill_raw_docs,
            })

    # Step 3: Never exceed the target.
    evidence_chunks = evidence_chunks[:TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION]

    # Save metadata inside pyrag_result for raw JSON/debug logs.
    pyrag_result["_initial_unique_evidence_chunks"] = initial_unique_count
    pyrag_result["_post_dedup_unique_fill_added"] = fill_added
    pyrag_result["_final_unique_evidence_chunks"] = len(evidence_chunks)
    pyrag_result["_post_dedup_unique_fill_warning"] = fill_warning

    if fill_warning:
        print("WARNING:", fill_warning)

    return {
        "type": sample.get("type"),
        "question": sample.get("question"),
        "answer": sample.get("answer"),
        "supports": sample.get("supports", []),
        "evidence_chunk": evidence_chunks,
    }


def get_sample_uid(sample):
    # Phase 1 added _pyrag_original_index before shuffling.
    # If it does not exist, fall back to the question string.
    if "_pyrag_original_index" in sample:
        return str(sample["_pyrag_original_index"])
    return sample.get("question", "")


def load_done_uids(progress_jsonl):
    rows = load_jsonl(progress_jsonl)
    done = set()

    for row in rows:
        uid = row.get("_pyrag_original_index")
        if uid is not None:
            done.add(str(uid))
        elif row.get("question"):
            done.add(row["question"])

    return done


def rebuild_final_json_from_progress(progress_jsonl, final_json):
    rows = load_jsonl(progress_jsonl)
    final_rows = []

    for row in rows:
        evidence_chunk = row.get("evidence_chunk", [])
        evidence_chunk = evidence_chunk[:TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION]

        clean = {
            "type": row.get("type"),
            "question": row.get("question"),
            "answer": row.get("answer"),
            "supports": row.get("supports", []),
            "evidence_chunk": evidence_chunk,
        }
        final_rows.append(clean)

    atomic_write_json(final_json, final_rows)
    return final_rows


print("Evidence helper functions are ready.")
print("Target unique evidence chunks per question:", TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)
print("Post-dedup unique fill enabled:", ENABLE_POST_DEDUP_UNIQUE_FILL)
print("Post-dedup fill candidate topk:", POST_DEDUP_FILL_CANDIDATE_TOPK)

Evidence helper functions are ready.
Target unique evidence chunks per question: 6
Post-dedup unique fill enabled: True
Post-dedup fill candidate topk: 50


In [ ]:
# cell 11.5
# Reset old Phase 2 outputs before starting the raw-top-6 run.
#
# Keep RESET_PHASE2_OUTPUTS = True when you want to rerun from the beginning.
# This is important because old progress JSONL files may contain top-5 rows.
# If those rows remain, later cells will skip them.

RESET_PHASE2_OUTPUTS = True


def phase2_output_paths_for_dataset(dataset_name):
    cfg = DATASETS[dataset_name]
    return [
        cfg["raw_jsonl"],
        cfg["progress_jsonl"],
        cfg["final_evidence"],
        f"{RAW_OUTPUT_DIR}/{dataset_name}_errors.jsonl",
        f"{RAW_OUTPUT_DIR}/{dataset_name}_quiet_run.log",
        f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_errors.jsonl",
        f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_success.jsonl",
        f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_quiet_run.log",
    ]


if RESET_PHASE2_OUTPUTS:
    deleted = []
    for dataset_name in DATASETS:
        for path in phase2_output_paths_for_dataset(dataset_name):
            p = Path(path)
            if p.exists():
                p.unlink()
                deleted.append(str(p))

    print("RESET_PHASE2_OUTPUTS is True.")
    print("Deleted old Phase 2 files:", len(deleted))
    for path in deleted:
        print("  -", path)
else:
    print("RESET_PHASE2_OUTPUTS is False.")
    print("Existing progress files will be reused.")
    print("Warning: old top-5 or capped rows may remain and be skipped by later cells.")

RESET_PHASE2_OUTPUTS is True.
Deleted old Phase 2 files: 16
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_pyrag_raw_results.jsonl
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_evidence_progress.jsonl
  - /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_errors.jsonl
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_quiet_run.log
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_errors.jsonl
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_success.jsonl
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_quiet_run.log
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_pyrag_raw_results.jsonl
  - /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_evidence_progress.jsonl
  - /con

In [ ]:
# cell 12
# Smoke-test one PyRAG question on HotpotQA without writing final outputs.
# This checks that vLLM + retriever + raw-top-6 PyRAG + post-dedup unique fill work together.
#
# This cell does not raise an error if final unique evidence_chunk is below 6.
# It reports what happened.

SMOKE_DATASET = "hotpotqa"

start_retriever_for_dataset(SMOKE_DATASET)
runner = build_pyrag_runner()

smoke_samples = load_json_array(DATASETS[SMOKE_DATASET]["prepared_questions"])
smoke_sample = smoke_samples[0]
smoke_question = smoke_sample["question"]

print("Smoke-test dataset:", SMOKE_DATASET)
print("Smoke-test question:", smoke_question)

smoke_result = runner.run(smoke_question, topk=DEFAULT_TOPK)

smoke_evidence_item = build_final_evidence_item(
    sample=smoke_sample,
    pyrag_result=smoke_result,
)

smoke_evidence_unique = smoke_evidence_item["evidence_chunk"]

smoke_evidence_raw = extract_evidence_chunks_from_execution_log(
    smoke_result["execution_log"],
    max_chunks=TARGET_RAW_RETRIEVED_CHUNKS_PER_QUESTION,
    deduplicate=False,
)

num_raw_retrieved_from_log = count_retrieved_docs_in_execution_log(smoke_result["execution_log"])
num_unique_retrieved_from_log = count_unique_retrieved_docs_in_execution_log(smoke_result["execution_log"])

print("\nSmoke-test final answer:", smoke_result.get("final_answer"))
print("Planned retrieve calls:", smoke_result.get("planned_retrieve_calls"))
print("Budget allocations:", smoke_result.get("retrieval_budget_allocations"))
print("RAW retrieved chunks in retrieve steps:", num_raw_retrieved_from_log)
print("Unique retrieved chunks in retrieve steps:", num_unique_retrieved_from_log)
print("Initial unique evidence chunks:", smoke_result.get("_initial_unique_evidence_chunks"))
print("Post-dedup unique fill added:", smoke_result.get("_post_dedup_unique_fill_added"))
print("Final unique evidence chunks:", len(smoke_evidence_unique))
print("Target unique evidence chunks:", TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)

if smoke_result.get("_post_dedup_unique_fill_warning"):
    print("WARNING:", smoke_result["_post_dedup_unique_fill_warning"])

if len(smoke_evidence_unique) < TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION:
    print(
        "NOTE: final evidence_chunk is still below target because the retriever "
        "did not return enough non-duplicate candidates from the original question."
    )

print("\nRAW retrieved evidence preview:")
for i, ev in enumerate(smoke_evidence_raw, 1):
    print("-" * 80)
    print(f"Raw Evidence {i}: {ev['title']}")
    print(ev["text"][:500])

print("\nFinal unique evidence preview:")
for i, ev in enumerate(smoke_evidence_unique, 1):
    print("-" * 80)
    print(f"Unique Evidence {i}: {ev['title']}")
    print(ev["text"][:500])

Killed process tree: 9676
Stopped previous retriever process.
Starting retriever for: hotpotqa
Index dir: /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa
Command: /usr/bin/python3 /content/drive/MyDrive/final_project/pyrag/custom_scripts/local_e5_retriever_server.py --index_dir /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa --model_name intfloat/e5-base-v2 --device cpu --host 127.0.0.1 --port 8008
Log: /content/drive/MyDrive/final_project/pyrag/runs/retriever_hotpotqa.log
Retriever health: {'status': 'ok', 'num_documents': 35029, 'index_vectors': 35029, 'model_name': 'intfloat/e5-base-v2', 'device': 'cpu'}
Smoke-test dataset: hotpotqa
Smoke-test question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?

=== Sub-queries ===
  1. When was Bedknobs and Broomsticks released?
  2. When was The Muppet Christmas Carol released?

=== Generated Code ===
docs1 = retrieve("When was Bedknobs and Broomsticks released?")
release1 

In [ ]:
# cell 13
# Main function to run one dataset end-to-end.
#
# Quiet mode:
# - Notebook output shows tqdm and short dataset-level messages only.
# - PyRAG internal prints are redirected to a per-dataset log file.
# - Final JSON is rebuilt from progress JSONL at checkpoints and at the end.
#
# Required final outputs:
# - /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
# - /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json

import contextlib
import logging
from datetime import datetime, timezone
from tqdm.auto import tqdm


def utc_now_iso():
    # Return timezone-aware UTC timestamp.
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def run_pyrag_quiet(runner, question, topk, log_path, sample_uid, dataset_name):
    # Run PyRAG while redirecting verbose stdout/stderr to a log file.
    # This keeps the notebook output clean.
    with open(log_path, "a", encoding="utf-8") as log_f:
        log_f.write("\n" + "=" * 120 + "\n")
        log_f.write(f"dataset={dataset_name} uid={sample_uid} time={utc_now_iso()}\n")
        log_f.write(f"question={question}\n")
        log_f.write("=" * 120 + "\n")

        with contextlib.redirect_stdout(log_f), contextlib.redirect_stderr(log_f):
            result = runner.run(question, topk=topk)

    return result


def run_dataset_end_to_end(dataset_name, max_samples=None):
    if dataset_name not in DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    cfg = DATASETS[dataset_name]
    quiet_log_path = f"{RAW_OUTPUT_DIR}/{dataset_name}_quiet_run.log"
    error_path = f"{RAW_OUTPUT_DIR}/{dataset_name}_errors.jsonl"

    print("=" * 100)
    print("Running dataset:", dataset_name)
    print("Prepared questions:", cfg["prepared_questions"])
    print("Raw output JSONL:", cfg["raw_jsonl"])
    print("Progress evidence JSONL:", cfg["progress_jsonl"])
    print("Final evidence JSON:", cfg["final_evidence"])
    print("Quiet PyRAG log:", quiet_log_path)

    # Start the dataset-specific retriever.
    start_retriever_for_dataset(dataset_name)

    # Build a fresh runner after the retriever is ready.
    runner = build_pyrag_runner()

    samples = load_json_array(cfg["prepared_questions"])
    if max_samples is not None:
        samples = samples[:max_samples]

    selected_uids = {get_sample_uid(s) for s in samples}
    done_uids = load_done_uids(cfg["progress_jsonl"])
    completed_before = len(done_uids & selected_uids)

    print("Total samples selected:", len(samples))
    print("Already completed selected samples:", completed_before)

    completed_this_run = 0
    failed_this_run = 0
    skipped_this_run = 0

    # Reduce notebook noise from PyRAG logger if any.
    pyrag_logger = logging.getLogger("pyrag")
    old_pyrag_level = pyrag_logger.level
    pyrag_logger.setLevel(logging.ERROR)

    pbar = tqdm(samples, desc=f"PyRAG {dataset_name}", total=len(samples))

    try:
        for sample in pbar:
            uid = get_sample_uid(sample)

            if uid in done_uids:
                skipped_this_run += 1
                pbar.set_postfix(
                    ok=completed_this_run,
                    err=failed_this_run,
                    skip=skipped_this_run,
                    refresh=False,
                )
                continue

            question = sample["question"]

            try:
                # This is the only input sent to PyRAG.
                # No gold titles/docs/chunks/answer/supports are passed to PyRAG retrieval.
                pyrag_result = run_pyrag_quiet(
                    runner=runner,
                    question=question,
                    topk=DEFAULT_TOPK,
                    log_path=quiet_log_path,
                    sample_uid=uid,
                    dataset_name=dataset_name,
                )

                # Build final evidence first.
                # This may perform post-dedup unique fill using only sample["question"].
                evidence_item = build_final_evidence_item(sample, pyrag_result)

                # Save raw PyRAG result after post-dedup fill metadata/trace has been added.
                raw_record = {
                    "dataset": dataset_name,
                    "_pyrag_original_index": sample.get("_pyrag_original_index"),
                    "question": question,
                    "answer": sample.get("answer"),
                    "type": sample.get("type"),
                    "pyrag_result": pyrag_result,
                    "created_at": utc_now_iso(),
                }
                append_jsonl(cfg["raw_jsonl"], raw_record)

                # Add internal checkpoint/debug keys only to progress JSONL.
                progress_item = dict(evidence_item)
                progress_item["_dataset"] = dataset_name
                progress_item["_pyrag_original_index"] = sample.get("_pyrag_original_index")
                progress_item["_pyrag_final_answer"] = pyrag_result.get("final_answer")
                progress_item["_retried_with_topk10"] = pyrag_result.get("retried_with_topk10", False)
                progress_item["_num_evidence_chunks"] = len(evidence_item["evidence_chunk"])
                progress_item["_initial_unique_evidence_chunks"] = pyrag_result.get("_initial_unique_evidence_chunks")
                progress_item["_post_dedup_unique_fill_added"] = pyrag_result.get("_post_dedup_unique_fill_added")
                progress_item["_final_unique_evidence_chunks"] = pyrag_result.get("_final_unique_evidence_chunks")
                progress_item["_post_dedup_unique_fill_warning"] = pyrag_result.get("_post_dedup_unique_fill_warning")
                progress_item["_created_at"] = utc_now_iso()

                append_jsonl(cfg["progress_jsonl"], progress_item)

                done_uids.add(uid)
                completed_this_run += 1

                # Keep final JSON valid during long runs.
                if completed_this_run % SAVE_EVERY == 0:
                    rebuild_final_json_from_progress(
                        cfg["progress_jsonl"],
                        cfg["final_evidence"],
                    )

            except Exception as e:
                failed_this_run += 1

                error_record = {
                    "dataset": dataset_name,
                    "_pyrag_original_index": sample.get("_pyrag_original_index"),
                    "question": question,
                    "error_type": type(e).__name__,
                    "error": str(e),
                    "created_at": utc_now_iso(),
                }
                append_jsonl(error_path, error_record)

                # Store full traceback in quiet log, not notebook.
                with open(quiet_log_path, "a", encoding="utf-8") as log_f:
                    log_f.write("\n" + "!" * 120 + "\n")
                    log_f.write(f"ERROR dataset={dataset_name} uid={uid} time={utc_now_iso()}\n")
                    log_f.write(traceback.format_exc())
                    log_f.write("\n" + "!" * 120 + "\n")

                # Continue with the next sample.
                continue

            finally:
                pbar.set_postfix(
                    ok=completed_this_run,
                    err=failed_this_run,
                    skip=skipped_this_run,
                    refresh=False,
                )

    finally:
        pyrag_logger.setLevel(old_pyrag_level)

    # Always rebuild the final JSON at the end.
    final_rows = rebuild_final_json_from_progress(
        cfg["progress_jsonl"],
        cfg["final_evidence"],
    )

    summary = {
        "dataset": dataset_name,
        "total_selected": len(samples),
        "completed_before": completed_before,
        "completed_this_run": completed_this_run,
        "failed_this_run": failed_this_run,
        "skipped_this_run": skipped_this_run,
        "final_rows": len(final_rows),
        "final_evidence": cfg["final_evidence"],
        "raw_jsonl": cfg["raw_jsonl"],
        "progress_jsonl": cfg["progress_jsonl"],
        "error_jsonl": error_path,
        "quiet_log": quiet_log_path,
    }

    print("=" * 100)
    print("Finished dataset:", dataset_name)
    print("Completed this run:", completed_this_run)
    print("Failed this run:", failed_this_run)
    print("Skipped this run:", skipped_this_run)
    print("Final evidence rows:", len(final_rows))
    print("Final evidence file:", cfg["final_evidence"])

    return summary


print("Main dataset runner is ready.")

Main dataset runner is ready.


In [14]:
# cell 14
# Run HotpotQA separately.
# Notebook output should mainly be tqdm.

if RUN_HOTPOTQA:
    hotpotqa_summary = run_dataset_end_to_end(
        dataset_name="hotpotqa",
        max_samples=MAX_SAMPLES_PER_DATASET,
    )
else:
    print("RUN_HOTPOTQA is False; skipping HotpotQA.")

Running dataset: hotpotqa
Prepared questions: /content/drive/MyDrive/final_project/pyrag/prepared_questions/hotpotqa_shuffled.json
Raw output JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_pyrag_raw_results.jsonl
Progress evidence JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_evidence_progress.jsonl
Final evidence JSON: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
Quiet PyRAG log: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_quiet_run.log
Killed process tree: 9985
Stopped previous retriever process.
Starting retriever for: hotpotqa
Index dir: /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa
Command: /usr/bin/python3 /content/drive/MyDrive/final_project/pyrag/custom_scripts/local_e5_retriever_server.py --index_dir /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa --model_name intfloat/e5-base-v2 --device cpu --host 127.0.0.1 --port 8008
Log: /content/drive/MyDr

PyRAG hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: hotpotqa
Completed this run: 994
Failed this run: 6
Skipped this run: 0
Final evidence rows: 994
Final evidence file: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json


In [15]:
# cell 15
# Run 2WikiMultihopQA separately.
# This switches the retriever to the 2Wiki index.

if RUN_2WIKIMULTIHOPQA:
    wiki_summary = run_dataset_end_to_end(
        dataset_name="2wikimultihopqa",
        max_samples=MAX_SAMPLES_PER_DATASET,
    )
else:
    print("RUN_2WIKIMULTIHOPQA is False; skipping 2WikiMultihopQA.")

Running dataset: 2wikimultihopqa
Prepared questions: /content/drive/MyDrive/final_project/pyrag/prepared_questions/2wikimultihopqa_shuffled.json
Raw output JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_pyrag_raw_results.jsonl
Progress evidence JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_evidence_progress.jsonl
Final evidence JSON: /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json
Quiet PyRAG log: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_quiet_run.log
Killed process tree: 10283
Stopped previous retriever process.
Starting retriever for: 2wikimultihopqa
Index dir: /content/drive/MyDrive/final_project/pyrag/indexes/2wikimultihopqa
Command: /usr/bin/python3 /content/drive/MyDrive/final_project/pyrag/custom_scripts/local_e5_retriever_server.py --index_dir /content/drive/MyDrive/final_project/pyrag/indexes/2wikimultihopqa --model_name intfloat/e5-base-v2 --

PyRAG 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: 2wikimultihopqa
Completed this run: 998
Failed this run: 2
Skipped this run: 0
Final evidence rows: 998
Final evidence file: /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json


In [16]:
# cell 16A
# Retry failed HotpotQA samples only.
#
# Run this after cell 14 and cell 15 are finished.
#
# This cell:
# 1. Reads hotpotqa_errors.jsonl.
# 2. Shows what the errors were.
# 3. Finds failed HotpotQA samples not already saved in progress JSONL.
# 4. Re-runs only those samples.
# 5. Appends successful retries to progress JSONL.
# 6. Rebuilds hotpotqa_evidence.json in the original prepared_questions order.

from collections import Counter
from tqdm.auto import tqdm
import traceback
import json
from pathlib import Path

RETRY_TOPK = DEFAULT_TOPK
MAX_RETRY_ROUNDS_HOTPOTQA = 2

def retry_error_path(dataset_name):
    return f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_errors.jsonl"

def retry_success_path(dataset_name):
    return f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_success.jsonl"

def original_error_path(dataset_name):
    return f"{RAW_OUTPUT_DIR}/{dataset_name}_errors.jsonl"

def load_error_rows_for_dataset(dataset_name):
    # Load original and retry errors.
    rows = []
    for path in [original_error_path(dataset_name), retry_error_path(dataset_name)]:
        p = Path(path)
        if p.exists():
            rows.extend(load_jsonl(p))
    return rows

def summarize_dataset_errors(dataset_name, max_examples=10):
    # Print compact error summary.
    rows = load_error_rows_for_dataset(dataset_name)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Error rows found:", len(rows))

    if not rows:
        print("No error rows found.")
        return []

    latest_by_uid = {}
    for row in rows:
        uid = row.get("_pyrag_original_index")
        if uid is not None:
            uid = str(uid)
        else:
            uid = row.get("question", "")

        if uid:
            latest_by_uid[uid] = row

    latest_rows = list(latest_by_uid.values())
    print("Unique failed sample IDs:", len(latest_rows))

    grouped = Counter(
        (
            row.get("error_type", "UnknownError"),
            str(row.get("error", ""))[:300],
        )
        for row in latest_rows
    )

    print("\nError groups:")
    for (error_type, error_msg), count in grouped.most_common():
        print("-" * 100)
        print("Count:", count)
        print("Type:", error_type)
        print("Message:", error_msg)

    print("\nExamples:")
    for row in latest_rows[:max_examples]:
        print("-" * 100)
        print("uid:", row.get("_pyrag_original_index"))
        print("type:", row.get("error_type"))
        print("error:", str(row.get("error", ""))[:500])
        print("question:", str(row.get("question", ""))[:500])

    return latest_rows

def build_uid_to_sample(dataset_name):
    # Map prepared question uid to full original sample.
    cfg = DATASETS[dataset_name]
    samples = load_json_array(cfg["prepared_questions"])

    uid_to_sample = {}
    for sample in samples:
        uid_to_sample[get_sample_uid(sample)] = sample

    return uid_to_sample

def load_progress_rows_by_uid(progress_jsonl):
    # Keep latest progress row per uid.
    rows = load_jsonl(progress_jsonl)
    by_uid = {}

    for row in rows:
        uid = row.get("_pyrag_original_index")
        if uid is not None:
            uid = str(uid)
        else:
            uid = row.get("question", "")

        if uid:
            by_uid[uid] = row

    return by_uid

def rebuild_final_json_from_progress_ordered(dataset_name):
    # Rebuild final JSON in prepared_questions order.
    cfg = DATASETS[dataset_name]

    samples = load_json_array(cfg["prepared_questions"])
    progress_by_uid = load_progress_rows_by_uid(cfg["progress_jsonl"])

    final_rows = []
    missing_uids = []

    for sample in samples:
        uid = get_sample_uid(sample)
        row = progress_by_uid.get(uid)

        if row is None:
            missing_uids.append(uid)
            continue

        clean = {
            "type": row.get("type"),
            "question": row.get("question"),
            "answer": row.get("answer"),
            "supports": row.get("supports", []),
            "evidence_chunk": row.get("evidence_chunk", []),
        }

        final_rows.append(clean)

    atomic_write_json(cfg["final_evidence"], final_rows)

    print("=" * 100)
    print("Rebuilt ordered final JSON:", dataset_name)
    print("Final file:", cfg["final_evidence"])
    print("Final rows:", len(final_rows))
    print("Missing rows:", len(missing_uids))

    if missing_uids:
        print("First missing uids:", missing_uids[:20])

    return final_rows, missing_uids

def get_retry_candidates(dataset_name):
    # Failed samples that are not already completed in progress JSONL.
    cfg = DATASETS[dataset_name]

    error_rows = load_error_rows_for_dataset(dataset_name)
    done_uids = load_done_uids(cfg["progress_jsonl"])
    uid_to_sample = build_uid_to_sample(dataset_name)

    latest_error_by_uid = {}

    for row in error_rows:
        uid = row.get("_pyrag_original_index")
        if uid is not None:
            uid = str(uid)
        else:
            uid = row.get("question", "")

        if uid:
            latest_error_by_uid[uid] = row

    retry_samples = []
    missing_in_prepared = []

    for uid, err_row in latest_error_by_uid.items():
        if uid in done_uids:
            continue

        sample = uid_to_sample.get(uid)
        if sample is None:
            missing_in_prepared.append(uid)
            continue

        retry_samples.append(sample)

    print("=" * 100)
    print("Retry candidates:", dataset_name)
    print("Error uids:", len(latest_error_by_uid))
    print("Already completed among errors:", len(set(latest_error_by_uid) & done_uids))
    print("Will retry:", len(retry_samples))
    print("Missing in prepared_questions:", len(missing_in_prepared))

    if missing_in_prepared:
        print("First missing prepared uids:", missing_in_prepared[:20])

    return retry_samples

def retry_failed_samples_for_dataset(dataset_name, max_retry_rounds=2):
    # Retry only missing failed samples for one dataset.
    cfg = DATASETS[dataset_name]

    quiet_log_path = f"{RAW_OUTPUT_DIR}/{dataset_name}_retry_quiet_run.log"
    retry_err_path = retry_error_path(dataset_name)
    retry_ok_path = retry_success_path(dataset_name)

    print("=" * 100)
    print("Retry dataset:", dataset_name)
    print("Quiet retry log:", quiet_log_path)
    print("Retry error JSONL:", retry_err_path)
    print("Retry success JSONL:", retry_ok_path)

    # Show what failed before retrying.
    summarize_dataset_errors(dataset_name)

    retry_samples = get_retry_candidates(dataset_name)
    if not retry_samples:
        print("No failed samples need retry for:", dataset_name)
        final_rows, missing_uids = rebuild_final_json_from_progress_ordered(dataset_name)
        return {
            "dataset": dataset_name,
            "retried_success": 0,
            "retried_failed": 0,
            "remaining_missing": len(missing_uids),
            "final_rows": len(final_rows),
        }

    # Start correct retriever for this dataset.
    start_retriever_for_dataset(dataset_name)

    # Build fresh runner after retriever switch.
    runner = build_pyrag_runner()

    total_success = 0
    total_failed = 0

    for round_idx in range(1, max_retry_rounds + 1):
        retry_samples = get_retry_candidates(dataset_name)

        if not retry_samples:
            print("All failed samples recovered before round:", round_idx)
            break

        print("=" * 100)
        print(f"Retry round {round_idx}/{max_retry_rounds}")
        print("Samples to retry:", len(retry_samples))

        pbar = tqdm(retry_samples, desc=f"Retry {dataset_name} r{round_idx}")

        for sample in pbar:
            uid = get_sample_uid(sample)

            # Skip if completed during this retry run.
            done_uids = load_done_uids(cfg["progress_jsonl"])
            if uid in done_uids:
                continue

            question = sample["question"]

            try:
                pyrag_result = run_pyrag_quiet(
                    runner=runner,
                    question=question,
                    topk=RETRY_TOPK,
                    log_path=quiet_log_path,
                    sample_uid=uid,
                    dataset_name=dataset_name,
                )

                raw_record = {
                    "dataset": dataset_name,
                    "_pyrag_original_index": sample.get("_pyrag_original_index"),
                    "question": question,
                    "answer": sample.get("answer"),
                    "type": sample.get("type"),
                    "pyrag_result": pyrag_result,
                    "created_at": utc_now_iso(),
                    "_retry_round": round_idx,
                }
                append_jsonl(cfg["raw_jsonl"], raw_record)

                evidence_item = build_final_evidence_item(sample, pyrag_result)

                progress_item = dict(evidence_item)
                progress_item["_dataset"] = dataset_name
                progress_item["_pyrag_original_index"] = sample.get("_pyrag_original_index")
                progress_item["_pyrag_final_answer"] = pyrag_result.get("final_answer")
                progress_item["_retried_with_topk10"] = pyrag_result.get("retried_with_topk10", False)
                progress_item["_num_evidence_chunks"] = len(evidence_item["evidence_chunk"])
                progress_item["_created_at"] = utc_now_iso()
                progress_item["_recovered_from_error"] = True
                progress_item["_retry_round"] = round_idx

                append_jsonl(cfg["progress_jsonl"], progress_item)

                append_jsonl(
                    retry_ok_path,
                    {
                        "dataset": dataset_name,
                        "_pyrag_original_index": sample.get("_pyrag_original_index"),
                        "question": question,
                        "created_at": utc_now_iso(),
                        "_retry_round": round_idx,
                    },
                )

                total_success += 1

            except Exception as e:
                total_failed += 1

                append_jsonl(
                    retry_err_path,
                    {
                        "dataset": dataset_name,
                        "_pyrag_original_index": sample.get("_pyrag_original_index"),
                        "question": question,
                        "error_type": type(e).__name__,
                        "error": str(e),
                        "created_at": utc_now_iso(),
                        "_retry_round": round_idx,
                    },
                )

                with open(quiet_log_path, "a", encoding="utf-8") as log_f:
                    log_f.write("\n" + "!" * 120 + "\n")
                    log_f.write(
                        f"RETRY ERROR dataset={dataset_name} uid={uid} "
                        f"round={round_idx} time={utc_now_iso()}\n"
                    )
                    log_f.write(traceback.format_exc())
                    log_f.write("\n" + "!" * 120 + "\n")

            finally:
                pbar.set_postfix(
                    ok=total_success,
                    err=total_failed,
                    refresh=False,
                )

    final_rows, missing_uids = rebuild_final_json_from_progress_ordered(dataset_name)

    print("=" * 100)
    print("Retry finished:", dataset_name)
    print("Recovered successes:", total_success)
    print("Retry failures:", total_failed)
    print("Final rows:", len(final_rows))
    print("Remaining missing rows:", len(missing_uids))

    return {
        "dataset": dataset_name,
        "retried_success": total_success,
        "retried_failed": total_failed,
        "remaining_missing": len(missing_uids),
        "final_rows": len(final_rows),
    }

# Run retry only for HotpotQA.
hotpotqa_retry_summary = retry_failed_samples_for_dataset(
    dataset_name="hotpotqa",
    max_retry_rounds=MAX_RETRY_ROUNDS_HOTPOTQA,
)

print("=" * 100)
print("HotpotQA retry summary:")
print(json.dumps(hotpotqa_retry_summary, ensure_ascii=False, indent=2))

Retry dataset: hotpotqa
Quiet retry log: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_quiet_run.log
Retry error JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_errors.jsonl
Retry success JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_retry_success.jsonl
Dataset: hotpotqa
Error rows found: 6
Unique failed sample IDs: 6

Error groups:
----------------------------------------------------------------------------------------------------
Count: 2
Type: TypeError
Message: Object of type set is not JSON serializable
----------------------------------------------------------------------------------------------------
Count: 1
Type: RuntimeError
Message: Code execution failed (NameError: name 'collaborators2' is not defined)
--- Generated code ---
docs1 = retrieve("Who worked on the screenplay with Edward Carfagno?")
colaborators1 = answer("Who worked on the screenplay with Edward Carfagno?", docs1)

d

Retry hotpotqa r1:   0%|          | 0/6 [00:00<?, ?it/s]

Retry candidates: hotpotqa
Error uids: 6
Already completed among errors: 5
Will retry: 1
Missing in prepared_questions: 0
Retry round 2/2
Samples to retry: 1


Retry hotpotqa r2:   0%|          | 0/1 [00:00<?, ?it/s]

Rebuilt ordered final JSON: hotpotqa
Final file: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
Final rows: 1000
Missing rows: 0
Retry finished: hotpotqa
Recovered successes: 6
Retry failures: 1
Final rows: 1000
Remaining missing rows: 0
HotpotQA retry summary:
{
  "dataset": "hotpotqa",
  "retried_success": 6,
  "retried_failed": 1,
  "remaining_missing": 0,
  "final_rows": 1000
}


In [17]:
# cell 16B
# Retry failed 2WikiMultihopQA samples only.
#
# Run this after:
# - cell 15 is fully finished
# - cell 16A has been executed at least once, because it defines helper functions.
#
# This cell:
# 1. Reads 2wikimultihopqa_errors.jsonl.
# 2. Shows what the errors were.
# 3. Finds failed 2Wiki samples not already saved in progress JSONL.
# 4. Re-runs only those samples.
# 5. Appends successful retries to progress JSONL.
# 6. Rebuilds 2wikimultihopqa_evidence.json in the original prepared_questions order.

MAX_RETRY_ROUNDS_2WIKI = 2

wiki_retry_summary = retry_failed_samples_for_dataset(
    dataset_name="2wikimultihopqa",
    max_retry_rounds=MAX_RETRY_ROUNDS_2WIKI,
)

print("=" * 100)
print("2WikiMultihopQA retry summary:")
print(json.dumps(wiki_retry_summary, ensure_ascii=False, indent=2))

Retry dataset: 2wikimultihopqa
Quiet retry log: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_retry_quiet_run.log
Retry error JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_retry_errors.jsonl
Retry success JSONL: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/2wikimultihopqa_retry_success.jsonl
Dataset: 2wikimultihopqa
Error rows found: 2
Unique failed sample IDs: 2

Error groups:
----------------------------------------------------------------------------------------------------
Count: 1
Type: RuntimeError
Message: generate_code failed after 3 retries. Last error: invalid syntax (<generated>, line 1)
--- Last model output (truncated) ---
```python
# Retrieve information about John Corvinus's paternal grandfather
docs1 = retrieve("Who is John Corvinus's paternal grandfather?")
grandfather = answer("Who is John 
-----------------------------------------------------------------------------------------------

Retry 2wikimultihopqa r1:   0%|          | 0/2 [00:00<?, ?it/s]

Retry candidates: 2wikimultihopqa
Error uids: 2
Already completed among errors: 1
Will retry: 1
Missing in prepared_questions: 0
Retry round 2/2
Samples to retry: 1


Retry 2wikimultihopqa r2:   0%|          | 0/1 [00:00<?, ?it/s]

Rebuilt ordered final JSON: 2wikimultihopqa
Final file: /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json
Final rows: 1000
Missing rows: 0
Retry finished: 2wikimultihopqa
Recovered successes: 2
Retry failures: 1
Final rows: 1000
Remaining missing rows: 0
2WikiMultihopQA retry summary:
{
  "dataset": "2wikimultihopqa",
  "retried_success": 2,
  "retried_failed": 1,
  "remaining_missing": 0,
  "final_rows": 1000
}


In [18]:
# cell 17
# Validate final JSON files.
#
# This cell reports the final evidence distribution.
# It does not stop execution if some rows have fewer than 6 unique chunks.
#
# Expected behavior:
# - Most rows should have 6 unique evidence chunks.
# - If a row has fewer than 6, it means even post-dedup fill with the original question
#   could not find enough non-duplicate candidates.
# - Retrieval fill never uses answer/supports.

from collections import Counter


def validate_final_evidence_file(dataset_name):
    cfg = DATASETS[dataset_name]
    final_path = Path(cfg["final_evidence"])
    progress_path = Path(cfg["progress_jsonl"])

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Final file:", final_path)
    print("Progress file:", progress_path)

    if not final_path.exists():
        print("WARNING: Final evidence file does not exist yet.")
        return None

    rows = load_json_array(final_path)
    print("Rows:", len(rows))

    required_keys = {"type", "question", "answer", "supports", "evidence_chunk"}

    bad_key_rows = []
    bad_evidence_rows = []
    over_target_rows = []
    duplicate_rows = []
    evidence_lengths = []

    for i, row in enumerate(rows):
        if set(row.keys()) != required_keys:
            bad_key_rows.append(i)

        evidence = row.get("evidence_chunk")
        if not isinstance(evidence, list):
            bad_evidence_rows.append(i)
            continue

        evidence_lengths.append(len(evidence))

        if len(evidence) > TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION:
            over_target_rows.append((i, len(evidence)))

        seen = set()
        has_duplicate = False

        for doc in evidence:
            if not isinstance(doc, dict):
                bad_evidence_rows.append(i)
                continue

            key = (
                str(doc.get("title", "")).strip(),
                str(doc.get("text", "")).strip(),
            )

            if key in seen:
                has_duplicate = True

            seen.add(key)

        if has_duplicate:
            duplicate_rows.append(i)

    length_dist = Counter(evidence_lengths)
    min_evidence = min(evidence_lengths) if evidence_lengths else 0
    max_evidence = max(evidence_lengths) if evidence_lengths else 0
    below_target = sum(1 for x in evidence_lengths if x < TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)
    equal_target = sum(1 for x in evidence_lengths if x == TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)
    above_target = sum(1 for x in evidence_lengths if x > TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)

    print("Rows with wrong keys:", len(bad_key_rows))
    print("Rows with invalid evidence_chunk:", len(bad_evidence_rows))
    print("Min final unique evidence chunks:", min_evidence)
    print("Max final unique evidence chunks:", max_evidence)
    print("Evidence length distribution:", dict(sorted(length_dist.items())))
    print("Rows below target:", below_target)
    print("Rows equal target:", equal_target)
    print("Rows above target:", above_target)
    print("Rows with duplicate evidence in final JSON:", len(duplicate_rows))
    print("Target unique evidence chunks:", TARGET_UNIQUE_EVIDENCE_CHUNKS_PER_QUESTION)

    if progress_path.exists():
        progress_rows = load_jsonl(progress_path)
        fill_added = [
            row.get("_post_dedup_unique_fill_added", 0)
            for row in progress_rows
            if isinstance(row.get("_post_dedup_unique_fill_added", 0), int)
        ]
        fill_warnings = [
            row for row in progress_rows
            if row.get("_post_dedup_unique_fill_warning")
        ]

        print("Rows where post-dedup fill added chunks:", sum(1 for x in fill_added if x > 0))
        print("Total chunks added by post-dedup fill:", sum(fill_added))
        print("Rows with post-dedup fill warning:", len(fill_warnings))

        if fill_warnings:
            print("First fill warning:", fill_warnings[0].get("_post_dedup_unique_fill_warning"))

    if bad_key_rows:
        print("WARNING: rows with wrong keys. First indices:", bad_key_rows[:10])

    if bad_evidence_rows:
        print("WARNING: invalid evidence_chunk rows. First indices:", bad_evidence_rows[:10])

    if over_target_rows:
        print("WARNING: rows over target. First rows:", over_target_rows[:10])

    if duplicate_rows:
        print("WARNING: duplicate chunks found in final evidence. First rows:", duplicate_rows[:10])

    if rows:
        print("Example question:", rows[0]["question"])
        print("Example final unique evidence chunks:", len(rows[0]["evidence_chunk"]))

        if rows[0]["evidence_chunk"]:
            print("First evidence title:", rows[0]["evidence_chunk"][0].get("title"))
            print("First evidence preview:", rows[0]["evidence_chunk"][0].get("text", "")[:300])

    print("Validation report finished:", dataset_name)

    return {
        "dataset": dataset_name,
        "rows": len(rows),
        "min_evidence": min_evidence,
        "max_evidence": max_evidence,
        "length_distribution": dict(sorted(length_dist.items())),
        "below_target": below_target,
        "equal_target": equal_target,
        "above_target": above_target,
        "bad_key_rows": len(bad_key_rows),
        "bad_evidence_rows": len(bad_evidence_rows),
        "duplicate_rows": len(duplicate_rows),
    }


validation_reports = {}

for ds in ["hotpotqa", "2wikimultihopqa"]:
    report = validate_final_evidence_file(ds)
    validation_reports[ds] = report

print("=" * 100)
print("Validation reports:")
print(json.dumps(validation_reports, ensure_ascii=False, indent=2))

Dataset: hotpotqa
Final file: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
Progress file: /content/drive/MyDrive/final_project/pyrag/raw_pyrag_outputs/hotpotqa_evidence_progress.jsonl
Rows: 1000
Rows with wrong keys: 0
Rows with invalid evidence_chunk: 0
Min final unique evidence chunks: 6
Max final unique evidence chunks: 6
Evidence length distribution: {6: 1000}
Rows below target: 0
Rows equal target: 1000
Rows above target: 0
Rows with duplicate evidence in final JSON: 0
Target unique evidence chunks: 6
Rows where post-dedup fill added chunks: 423
Total chunks added by post-dedup fill: 730
Rows with post-dedup fill warning: 0
Example question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Example final unique evidence chunks: 6
First evidence title: Bedknobs and Broomsticks
First evidence preview: Bedknobs and Broomsticks is a 1971 British-American musical fantasy film produced by Walt Disney Productions and release